### pymc3による希土類Co合金の磁気相転移温度のベイズ線形回帰

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.mlab as mlab
%matplotlib inline


pymcにより回帰係数のゆらぎの程度を調べます。

観測量が線形回帰式+ガウシアンノイズ項
$$
T = (X,w) + \delta
$$
とします。

希土類Co合金キュリー温度のデータセットを用います。

In [ ]:
def get_data():
    filename = "../data/TC_ReCo_detail_descriptor.csv"
    df = pd.read_csv(filename)
    descriptor_names = ['C_R', 'vol_per_atom', 'f4', 'S4f', 'J4f']
    target_name = "Tc"
    return df, descriptor_names, target_name


g_df, g_descriptor_names, g_target_name = get_data()


線形回帰式が
y = (X,w)
となるように[1]の列を加えておく。

In [ ]:
g_X = g_df[g_descriptor_names].values
g_T = g_df[g_target_name].values


規格化

In [ ]:
from sklearn.preprocessing import StandardScaler
g_X = StandardScaler().fit_transform(g_X)


intercept部分を加える

In [ ]:
from copy import copy


def add_Xintercept(X, descriptor_names):
    one = np.ones(X.shape[0])
    X = np.concatenate([X, one.reshape(-1, 1)], axis=1)
    descriptor = copy(descriptor_names)
    descriptor.append("intercept")
    return X, descriptor


g_X, g_descriptor = add_Xintercept(g_X, g_descriptor_names)


In [ ]:
print(g_descriptor)
# interceptが加わった。


可視化

In [ ]:
plt.plot(g_X)


まず線形回帰しておく。

平均場解からのゆらぎを調べるので、初期値を求めておく。

In [ ]:
from sklearn.linear_model import LinearRegression


def fit_linearmodel(X, y):
    """fit X and y using the linear model
    and plot them

    Args:
        X (np.array): descriptor
        y (np.array): target values

    Returns:
        np.array: coefficents of the linear model
    """
    reg = LinearRegression(fit_intercept=False)
    reg.fit(X, y)
    print("coef=", reg.coef_.ravel(), reg.intercept_)
    print("R2=", reg.score(X, y))

    # predictと可視化
    yp = reg.predict(X)

    # 図の最大最小
    yall = np.concatenate([y, yp])
    ylim = (yall.min(), yall.max())

    plt.figure(figsize=(5, 5))
    plt.plot(y, yp, "o")
    plt.plot(ylim, ylim)
    plt.xlabel("observed")
    plt.ylabel("predict")
    plt.show()
    return reg.coef_


g_linear_coef = fit_linearmodel(g_X, g_T)
g_linear_coef.shape


解が複数存在するので平均場付近を調べるために初期分布を線形回帰の解とする。


In [ ]:
import pymc as pm


def make_linear_model(X, T, descriptor, linear_coef):

    w_std = np.array([100, 10, 10, 10, 10, 100])*10.0

    sigma_std = 10

    X0 = X[:, 0]
    X1 = X[:, 1]
    X2 = X[:, 2]
    X3 = X[:, 3]
    X4 = X[:, 4]
    X5 = X[:, 5]

    basic_model = pm.Model()

    with basic_model:
        # arrayにはできないらしい。
        a0 = pm.Normal(descriptor[0], mu=linear_coef[0], sd=w_std[0])
        a1 = pm.Normal(descriptor[1], mu=linear_coef[1], sd=w_std[1])
        a2 = pm.Normal(descriptor[2], mu=linear_coef[2], sd=w_std[2])
        a3 = pm.Normal(descriptor[3], mu=linear_coef[3], sd=w_std[3])
        a4 = pm.Normal(descriptor[4], mu=linear_coef[4], sd=w_std[4])
        a5 = pm.Normal(descriptor[5], mu=linear_coef[5], sd=w_std[5])

        sigma = pm.HalfNormal('sigma', sd=sigma_std)

        # np.dot(X,w)には書けない。
        mu = a0*X0 + a1*X1 + a2*X2 + a3*X3 + a4*X4 + a5
        T_exp = pm.Normal('Y_exp', mu=mu, sd=sigma, observed=T)
    return basic_model


g_basic_modeL = make_linear_model(g_X, g_T, g_descriptor, g_linear_coef)


In [ ]:
g_basic_modeL


MCMCを用いて$w$の分布を出す。

まずはMAP解

In [ ]:
g_map_estimate = pm.find_MAP(model=g_basic_modeL)

g_map_estimate


MCによる解

In [ ]:
import pickle
import os
g_filename_trace = "linear_model_ReCo.pickle"
if not os.path.isfile(g_filename_trace):
    with g_basic_modeL:
        n_pm_sample = 5000
        g_trace = pm.sample(n_pm_sample)
        # it runs n_sample x jobs
    with open(g_filename_trace,"wb") as _f:
        pickle.dump(g_trace, _f)
    # virtualbox上のububutu,4core,minicondaでは約25秒で終了します。
    # 一方、Anacondaではかなり時間がかかりました。
else:
    with open(g_filename_trace,"rb") as _f:
        g_trace = pickle.load(_f)

In [ ]:
#_ = pm.plot_posterior(g_trace)
import arviz_plots as azp
azp.plot_dist(g_trace, group="posterior")

各変数ほぼ正規分布をしている。(後ろでgaussianでfitする。)

In [ ]:
#_ = pm.forestplot(g_trace)
azp.plot_forest(g_trace)


下図ではchain毎のhistgramを示す。

In [ ]:
#_ = pm.traceplot(g_trace)
azp.plot_trace(g_trace)

正規分布でfitした結果を示す。

線形回帰係数も表示する。

In [ ]:
#g_trace.varnames
list(g_trace.posterior.data_vars)

In [ ]:
def get_summary(g_trace):
    g_df_summary = pm.summary(g_trace).round(3)

    g_df_linearcoef = pd.DataFrame(
        g_linear_coef, index=g_descriptor, columns=["linear_reg"])
    g_df_linearcoef
    g_df_summary2 = pd.concat(
        [g_df_summary, g_df_linearcoef], axis=1, sort=False)
    return g_df_summary2

get_summary(g_trace)


In [ ]:
from numpy.random import multivariate_normal
from sklearn.mixture import GaussianMixture
import numpy as np
import matplotlib.pyplot as plt

def show_Y_Yp(trace, var, X, Y):
    """show posterior trace and predictions"""

    ylim = (Y.min(), Y.max())

    # MC sample の取得
    alist = []
    for a in var:
        v = trace.posterior[a].values.flatten()
        alist.append(v)

    # shape: (sample数, 変数数)
    a_distribution = np.array(alist).T
    print(a_distribution.shape)

    # mean and covariance fit
    cls = GaussianMixture(n_components=1)
    cls.fit(a_distribution)

    print("means", cls.means_[0])
    print("covariances", cls.covariances_[0])

    w_mean = cls.means_[0]
    print("w", w_mean)

    Y_mean = np.dot(X, w_mean)

    plt.figure(figsize=(5, 5))
    plt.plot(Y, Y_mean, "o")
    plt.ylim(ylim)
    plt.xlim(ylim)
    plt.plot(ylim, ylim)
    plt.xlabel("Tc exp.")
    plt.ylabel("Tc pred.")
    plt.show()

    plt.figure(figsize=(5, 5))

    n = 200
    w_rand = multivariate_normal(
        cls.means_[0],
        cls.covariances_[0],
        size=n
    )

    Y_randlist = []
    for w_rand1 in w_rand:
        Y_rand = np.dot(X, w_rand1)
        Y_randlist.append(list(Y_rand))

    Y_randlist = np.array(Y_randlist).T

    for t, y_rand in zip(Y, Y_randlist):
        plt.plot(
            np.ones(n) * t,
            np.sort(y_rand),
            ".",
            color="red",
            alpha=0.02
        )

    plt.plot(ylim, ylim)
    plt.ylim(ylim)
    plt.xlim(ylim)
    plt.xlabel("Tc exp.")
    plt.ylabel("Tc pred.")
    plt.show()

    return cls
    
_ = show_Y_Yp(g_trace, g_descriptor, g_X, g_T)


autocorrelationの表示。

In [ ]:
# _ = pm.autocorrplot(g_trace)
azp.plot_autocorr(g_trace)

In [ ]:
# pm.energyplot(trace)
azp.plot_energy(g_trace)